In [ ]:
#@title Install
%%capture
!pip install gradio openai -q

In [ ]:
#@title API Key
import os
from getpass import getpass
k = getpass("OpenAI API key: ")
if k.strip(): os.environ["OPENAI_API_KEY"] = k; print("Key set!")
else: print("Demo mode.")

In [ ]:
#@title Tool Use Sandbox - Real Agent
import gradio as gr, os, json

TOOLS = [
    {"type": "function", "function": {"name": "calculator", "description": "Math calculations", "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
    {"type": "function", "function": {"name": "get_weather", "description": "Get weather for a city", "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
    {"type": "function", "function": {"name": "lookup_employee", "description": "Look up employee by name", "parameters": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]}}},
    {"type": "function", "function": {"name": "search_products", "description": "Search products", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}
]

def exec_tool(name, args):
    a = json.loads(args) if isinstance(args, str) else args
    if name == "calculator":
        try: return f"Result: {eval(a['expression'])}"
        except: return "Error"
    if name == "get_weather":
        w = {"san francisco": "62F Foggy", "new york": "45F Cloudy", "miami": "82F Sunny"}
        return w.get(a.get("city", "").lower(), "55F Clear")
    if name == "lookup_employee":
        e = {"john": "John Smith, Engineering, john@co.com", "sarah": "Sarah Lee, Product, sarah@co.com"}
        for k, v in e.items():
            if k in a.get("name", "").lower(): return v
        return "Not found"
    if name == "search_products":
        return "Products: Laptop $1299, Mouse $49, Monitor $399"
    return "Done"

def run_agent(task, max_iter):
    out = [f"# Agent Execution\n**Task:** {task}\n---\n"]
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        out.append("**[Demo]** Enter API key to see real agent.\n\nExample: calculator(85*0.15) -> $12.75")
        return "".join(out)
    
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    msgs = [{"role": "system", "content": "You are a helpful assistant with tools. Use them to complete tasks."}, {"role": "user", "content": task}]
    
    for i in range(max_iter):
        out.append(f"## Iteration {i+1}\n")
        r = client.chat.completions.create(model="gpt-4o-mini", messages=msgs, tools=TOOLS, max_tokens=400)
        m = r.choices[0].message
        if m.content: out.append(f"**Agent:** {m.content}\n")
        if not m.tool_calls: out.append("\n*Done.*"); break
        msgs.append(m)
        for tc in m.tool_calls:
            out.append(f"**Tool:** `{tc.function.name}`\n```\n{tc.function.arguments}\n```\n")
            result = exec_tool(tc.function.name, tc.function.arguments)
            out.append(f"**Result:** {result}\n")
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        out.append("---\n")
    return "".join(out)

TASKS = ["Calculate 15% tip on $85", "Weather in San Francisco?", "Find John's email", "Search for electronics under $100"]

with gr.Blocks(title="Tool Use Sandbox", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Tool Use Sandbox\n\nWatch a **real agent** use function calling.")
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("**Tools:** calculator, get_weather, lookup_employee, search_products")
            task = gr.Dropdown(TASKS, value=TASKS[0], label="Task", allow_custom_value=True)
            mi = gr.Slider(1, 5, value=3, step=1, label="Max Iterations")
            btn = gr.Button("Run Agent", variant="primary")
        with gr.Column(scale=2):
            out = gr.Markdown("Select a task.")
    btn.click(run_agent, [task, mi], out)

In [ ]:
#@title Launch
demo.launch(share=True)